# 06 — Projet complet : pipeline de bout en bout

Ce notebook assemble **tout ce qui a été vu** dans les notebooks précédents,
sous forme d'un mini-projet réaliste :

> **Scénario :** une boutique en ligne stocke ses commandes dans SQLite (en local,
> mais le même code fonctionne avec Supabase/PostgreSQL/MySQL en changeant juste
> le connecteur). On veut produire un **tableau de bord de ventes** : connexion,
> requêtage, nettoyage, analyse, visualisation, export ET alerte automatique
> si une anomalie est détectée.

C'est exactement le pipeline **CONNEXION → REQUÊTAGE → EXPLOITATION → ACTION**
que ce repo enseigne de bout en bout.


## Étape 0 — Préparer des données de démonstration

In [ ]:
import sys
sys.path.append("..")

from sqlalchemy import text
from db_connection.sqlite_connector import get_sqlite_engine
from queries.write_queries import bulk_insert
import random
from datetime import datetime, timedelta

engine = get_sqlite_engine()

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS orders_demo"))
    conn.execute(text('''
        CREATE TABLE orders_demo (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            product TEXT,
            category TEXT,
            quantity INTEGER,
            unit_price REAL,
            order_date TEXT
        )
    '''))

produits = [
    ("Clavier mécanique", "Informatique", 89.99),
    ("Souris sans fil", "Informatique", 29.99),
    ("Écran 27 pouces", "Informatique", 249.0),
    ("Chaise de bureau", "Mobilier", 149.0),
    ("Bureau réglable", "Mobilier", 320.0),
]

rows = []
start = datetime(2026, 1, 1)
for i in range(60):
    product, category, price = random.choice(produits)
    qty = random.randint(1, 5)
    # on injecte volontairement 2 anomalies de quantité
    if i in (10, 40):
        qty = 50
    date = start + timedelta(days=random.randint(0, 59))
    rows.append({
        "product": product, "category": category,
        "quantity": qty, "unit_price": price,
        "order_date": date.strftime("%Y-%m-%d"),
    })

bulk_insert(engine, "orders_demo", rows)
print(f"{len(rows)} commandes de démonstration créées.")


## Étape 1 — CONNEXION

(déjà fait ci-dessus via `get_sqlite_engine()`).
Pour basculer sur Supabase/PostgreSQL/MySQL, il suffirait de remplacer cette
ligne par `get_supabase_sql_engine()`, `get_pg_engine()` ou `get_mysql_engine()`
— **tout le reste du pipeline ne change pas**, car il travaille sur un
`Engine` SQLAlchemy générique.


## Étape 2 — REQUÊTAGE

In [ ]:
from data_processing.result_to_dataframe import query_to_dataframe

df = query_to_dataframe(engine, "SELECT * FROM orders_demo")
print(f"{len(df)} commandes récupérées.")
df.head()


## Étape 3 — EXPLOITATION

In [ ]:
from data_processing.clean_transform import cast_types, add_computed_column
from data_processing.analyze_results import group_and_aggregate, top_n, detect_anomalies

df = cast_types(df, {"order_date": "datetime64[ns]", "quantity": "int64", "unit_price": "float64"})
df = add_computed_column(df, "total", lambda row: row["quantity"] * row["unit_price"])

ca_par_categorie = group_and_aggregate(df, group_by="category", agg_column="total", agg_func="sum")
top_produits = top_n(df.groupby("product")["total"].sum().reset_index(), column="total", n=3)

print("=== CA par catégorie ===")
print(ca_par_categorie.to_string(index=False))
print("\n=== Top 3 produits ===")
print(top_produits.to_string(index=False))


In [ ]:
# Détection d'anomalies sur les quantités commandées
anomalies = detect_anomalies(df, column="quantity", n_std=2.0)
print(f"{len(anomalies)} anomalie(s) détectée(s) :")
anomalies


## Étape 4 — VISUALISATION

In [ ]:
from data_processing.visualize_results import plot_bar_matplotlib

plot_bar_matplotlib(ca_par_categorie, x="category", y="total", title="Chiffre d'affaires par catégorie")


## Étape 5 — EXPORT & ACTION

On exporte le rapport final, et si des anomalies existent, on les enregistre
dans une table dédiée (simulation d'une alerte métier automatique).


In [ ]:
from data_processing.export_results import export_to_excel, export_to_table
from queries.write_queries import bulk_insert as bulk_insert_alerts

export_to_excel(ca_par_categorie, "../exports/dashboard_ventes.xlsx")
export_to_table(ca_par_categorie, engine, table_name="rapport_ca_categorie", if_exists="replace")

if not anomalies.empty:
    alerts = [
        {
            "product": row["product"],
            "message": f"Quantité anormale détectée : {row['quantity']} unités le {row['order_date'].date()}",
        }
        for _, row in anomalies.iterrows()
    ]
    with engine.begin() as conn:
        conn.execute(text('''
            CREATE TABLE IF NOT EXISTS order_alerts (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                product TEXT,
                message TEXT
            )
        '''))
    bulk_insert_alerts(engine, "order_alerts", alerts)
    print(f"⚠️  {len(alerts)} alerte(s) enregistrée(s) en base.")
else:
    print("Aucune anomalie, aucune alerte à créer.")

print("\n✅ Pipeline complet exécuté : connexion → requêtage → exploitation → action.")


## Récapitulatif du pipeline

| Étape | Module utilisé |
|---|---|
| 1. Connexion | `db_connection/*.py` |
| 2. Requêtage | `queries/*.py` |
| 3. Nettoyage/Transformation | `data_processing/clean_transform.py` |
| 4. Analyse | `data_processing/analyze_results.py` |
| 5. Visualisation | `data_processing/visualize_results.py` |
| 6. Export / Action | `data_processing/export_results.py` + `queries/write_queries.py` |

Tu as maintenant toutes les briques pour construire n'importe quel pipeline
de données, de la connexion à une base jusqu'à l'exploitation concrète du résultat.
